# Shapash Report

The Shapash Report feature allows data scientists to deliver to anyone interested in their project **a document that freezes different aspects of their work as a basis for an audit report**.

The shapash `generate_report` method generates an HTML report for your project.  
The report is generated as a single HTML file with embedded branding (including the logo), so the file can be moved and opened locally without breaking the logo.  
Some interactive resources may still rely on CDN assets depending on your environment, so an internet connection can still be required for full interactivity.  

The report contains the following information:
1. General information about the project
2. Description of the dataset used
3. Documentation about data preparation and feature engineering
4. Details about your model (library, parameters...)
5. Exploration of the data with a focus on the difference between train and test sets
6. Global explainability of the model
7. Model performance

The first three points are generated using a YAML file that the user should fill. An example is available [here](https://github.com/MAIF/shapash/blob/master/tutorial/report/config/project_information.yml).

This tutorial presents an example of how to generate the Shapash Report.

Content:
- Set up an example project
- Create and fill your project information that will be displayed in the report
- Generate the base Shapash Report
- Go further: generate a custom report with a custom YAML configuration

Data from Kaggle [House Prices](https://www.kaggle.com/c/house-prices-advanced-regression-techniques/data)

> Note: Open the generated HTML file in a browser to view the final report.

In [ ]:
import pandas as pd
from category_encoders import OrdinalEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split

## Building Supervized Model 

In [ ]:
from shapash.data.data_loader import data_loading
house_df, house_dict = data_loading('house_prices')
y_df=house_df['SalePrice']
X_df=house_df[house_df.columns.difference(['SalePrice'])]

In [ ]:
from category_encoders import OrdinalEncoder

categorical_features = [col for col in X_df.columns if X_df[col].dtype == 'object']

encoder = OrdinalEncoder(
    cols=categorical_features,
    handle_unknown='ignore',
    return_df=True).fit(X_df)

X_df = encoder.transform(X_df)

In [ ]:
Xtrain, Xtest, ytrain, ytest = train_test_split(X_df, y_df, train_size=0.75, random_state=1)

In [ ]:
regressor = RandomForestRegressor(n_estimators=50).fit(Xtrain, ytrain)

In [ ]:
y_pred = pd.DataFrame(regressor.predict(Xtest),columns=['pred'], index=Xtest.index)

## Fill your project information

**The next step is to create a YML file containing information about your project.**  

We will use the example file available [here](https://github.com/MAIF/shapash/blob/master/tutorial/report/utils/project_info.yml).  
**You are welcome to use this file as a template for your own report.**  

We display the information contained in the YML file below :

In [ ]:
import yaml

with open(r'config/project_information.yml') as file:
    project_info = yaml.full_load(file)

print(yaml.dump(project_info, sort_keys=False))

---
**If you want to create your own custom file :**

The keys of the YML file are the titles of the different sections in the report.  
The YML file must then respect the following format:

```yaml
Title of section 1:  
    property1 name: property1 value 
    property2 name: property2 value 
    ...
Title of section 2:  
    property1 name: property1 value 
    ...
```
> Note that the **date** can be computed automatically using the *auto* property value (see example above)

## Generate your report

### Declare and compile SmartExplainer object

In [ ]:
from shapash import SmartExplainer

In [ ]:
xpl = SmartExplainer(
    model=regressor,
    preprocessing=encoder, # Optional: compile step can use inverse_transform method
    features_dict=house_dict  # optional parameter, specifies label for features name 
) 

In [ ]:
xpl.compile(x=Xtest, y_pred=y_pred, y_target=ytest)

At this step the model can be checked and inspected using different methods of the SmartExplainer object we just created.  

Please refer to the other tutorials for more information.

### Generate the base Shapash Report

Next we can generate the report using the `generate_report` method of our SmartExplainer object.

We need to pass `x_train`, `y_train` and `y_test` parameters in order to explore the data used when training the model.

Please refer to the documentation for a full description of the parameters.


In [ ]:
xpl.generate_report(
    output_file='output/report.html', 
    project_info_file='config/project_information.yml',
    x_train=Xtrain,
    y_train=ytrain,
    y_test=ytest,
    title_story="House prices report",
    title_description="""This document is a data science report of the kaggle house prices tutorial project. 
        It was generated using the Shapash library.""",
    metrics=[
        {
            'path': 'sklearn.metrics.mean_absolute_error',
            'name': 'Mean absolute error', 
        },
        {
            'path': 'sklearn.metrics.mean_squared_error',
            'name': 'Mean squared error',
        }
    ]
)

> Note: `generate_report` no longer includes the deprecated `kernel_name` parameter.

## Customize your own report

Now let's customize our report by adding new sections.

To do so:
- Start from the default report YAML configuration.
- Remove, add, or reorder blocks depending on the sections you want.
- Pass your custom file with `yaml_path="path/to/your/custom/report.yml"` in the `generate_report` method.

For our simple example, we use [this custom YAML file](https://github.com/MAIF/shapash/blob/master/tutorial/generate_report/config/default_report_custom.yml).  
- It removes the multivariate analysis block from the dataset analysis.
- It adds sections **Relationship with target variable** and **Relationship between training variables** for this tutorial example.
- It includes additional elements in the metrics area.

Next, we use this YAML file to generate our custom report:

In [ ]:
xpl.generate_report(
    output_file='output/custom_report.html', 
    project_info_file='config/project_information.yml',
    x_train=Xtrain,
    y_train=ytrain,
    y_test=ytest,
    title_story="House prices report",
    title_description="""This document is a data science report of the kaggle house prices tutorial project. 
        It was generated using the Shapash library.""",
    metrics=[
        {
            'path': 'sklearn.metrics.mean_absolute_error',
            'name': 'Mean absolute error', 
        },
        {
            'path': 'sklearn.metrics.mean_squared_error',
            'name': 'Mean squared error',
        }
    ],
    working_dir='working',
    yaml_path='config/default_report_custom.yml'
 )